# Model B at 64×64 input and the Q6 comparison (Colab T4)

1. Trains Model B on Fashion-MNIST upscaled from 28×28 to 64×64, the same input the pretrained models (MobileNetV2, EfficientNetV2-B0) get. The upscaling uses the same `tf.image.resize` as the pretrained-model loader, keeping 1 channel scaled to 0–1. The architecture is unchanged (20,522 parameters).
2. Measures the cost of all four Q6 models (Model B at 28×28 and 64×64, MobileNetV2, EfficientNetV2-B0) **in this one session**, so CPU latency is comparable.
3. Builds the Q6 comparison table, taking each model's test accuracy from its results file.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells from top to bottom.

To save this notebook afterwards, use **File → Download → Download .ipynb**. Don't use "Save a copy in GitHub".

## 1. Google Drive (results survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/pattern_cnn_experiments/model_b_64")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Runs are saved under:", DRIVE_ROOT)

## 2. Get the code
Switch `BRANCH` to `"main"` once the branch has been merged.

In [ ]:
BRANCH = "feature/model-b-64"
REPO = Path("/content/CNN-for-Image-Classification")

%cd /content
if not REPO.exists():
    !git clone https://github.com/Seth724/CNN-for-Image-Classification.git
%cd /content/CNN-for-Image-Classification
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!git log --oneline -1

## 3. Check the GPU and data

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
assert tf.config.list_physical_devices("GPU"), "No GPU: Runtime > Change runtime type > T4 GPU, then rerun from the top."
!nvidia-smi --query-gpu=name --format=csv,noheader
!python scripts/verify_data.py

## 4. Smoke test (1 epoch, about a minute)
Check the summary shows `image (InputLayer) (None, 64, 64, 1)` and `Total params: 20,522`.

In [ ]:
!python scripts/train_model_b.py --input-size 64 --epochs 1 --optimizer rmsprop --learning-rate 0.003 --output-dir /content/smoke_b64

## 5. Helper
`run_b64(...)` trains one configuration into its own Drive folder and skips it if it already finished, so after a disconnect you can rerun everything.

In [ ]:
import json, subprocess

def run_b64(optimizer="rmsprop", lr=0.003, batch_size=128, epochs=60):
    name = f"model_b64_{optimizer}_lr{lr:g}_bs{batch_size}_e{epochs}"
    out = DRIVE_ROOT / name
    if (out / "metrics" / "evaluation.json").exists():
        print("skip", name, "(already finished)")
        return name
    cmd = (f"python scripts/train_model_b.py --input-size 64 --best-checkpoint "
           f"--optimizer {optimizer} --learning-rate {lr} --batch-size {batch_size} "
           f"--epochs {epochs} --seed 42 --output-dir '{out}'")
    print("run ", name)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out.mkdir(parents=True, exist_ok=True)
    (out / "train_log.txt").write_text(result.stdout + result.stderr)
    for line in result.stdout.splitlines():
        if line.startswith(("Training time", "Final validation", "Evaluated epoch", "Test accuracy")):
            print("     ", line)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{name} failed; see {out / 'train_log.txt'}")
    return name

## 6. Main run
Model B's best setting from Q3 (RMSprop, lr 0.003, batch 128), 60 epochs, evaluating the epoch with the lowest validation loss (the same selection rule as the pretrained models). About 5–10 minutes.

In [ ]:
run_b64("rmsprop", 0.003)

## 7. Optional: learning-rate check at 64×64
Larger images can shift the best learning rate, so this tries the neighbouring values.

In [ ]:
run_b64("rmsprop", 0.001)
run_b64("rmsprop", 0.002)

## 8. Model B runs, ranked by validation accuracy
Test accuracy is shown for reference only; the choice is made on validation.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
rows = []
for path in sorted(DRIVE_ROOT.glob("model_b64_*/metrics/evaluation.json")):
    e = json.loads(path.read_text()); r = e["reporting"]
    rows.append({
        "run": path.parent.parent.name,
        "selected_epoch": e["selected_epoch"],
        "best_val_accuracy": round(e["best_val_accuracy"], 4),
        "test_accuracy (report only)": round(e["test_accuracy"], 4),
        "params": e["total_parameters"],
        "macs_formula": e["total_macs"],
        "macs_profiler": r["estimated_macs_per_image"],
        "inference_size_kib": round(r["inference_model_size_mib"] * 1024, 1),
        "epoch_time_s": e["mean_epoch_time_s"],
    })
summary = pd.DataFrame(rows).sort_values("best_val_accuracy", ascending=False)
summary.to_csv(DRIVE_ROOT / "model_b64_runs.csv", index=False)
BEST = summary.iloc[0].run
print("Selected on validation:", BEST)
summary

## 9. Report figures for the selected run (same four figures as Q4)

In [ ]:
import matplotlib.pyplot as plt, numpy as np, seaborn as sns
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
run_dir = DRIVE_ROOT / BEST
fig_dir = run_dir / "report_figures"; fig_dir.mkdir(exist_ok=True)
h = pd.read_csv(run_dir / "metrics" / "history.csv")
m = json.loads((run_dir / "metrics" / "evaluation.json").read_text())

for key, label, fname in [("loss", "Loss", "loss_curves.png"), ("accuracy", "Accuracy", "accuracy_curves.png")]:
    plt.figure(figsize=(8, 5))
    plt.plot(h["epoch"], h[key], label=f"Training {label.lower()}", linewidth=2)
    plt.plot(h["epoch"], h[f"val_{key}"], label=f"Validation {label.lower()}", linewidth=2)
    plt.axvline(m["selected_epoch"], color="grey", linestyle="--", linewidth=1, label=f"selected epoch ({m['selected_epoch']})")
    plt.xlabel("Epoch"); plt.ylabel(label); plt.title(f"Model B, 64×64 input: {label.lower()} curves")
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.savefig(fig_dir / fname, dpi=300); plt.close()

plt.figure(figsize=(9, 7))
sns.heatmap(np.array(m["confusion_matrix"]), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title("Model B, 64×64 input: test confusion matrix")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.savefig(fig_dir / "confusion_matrix.png", dpi=300); plt.close()

pc = pd.DataFrame(m["per_class"]); pc["class"] = class_names
pc.set_index("class")[["precision", "recall", "f1_score"]].plot(kind="bar", figsize=(12, 6), ylim=(0, 1))
plt.title("Model B, 64×64 input: per-class metrics"); plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.savefig(fig_dir / "per_class_metrics.png", dpi=300); plt.close()
print("Figures saved to", fig_dir)

## 10. Q6 cost: all four models, measured in this session

Parameters, model size, compute (profiler MACs/FLOPs) and single-image CPU latency. Every model is built exactly as its training script builds it. Cost doesn't depend on the trained weight values, so untrained copies give the same numbers. Measuring all four in one session keeps the CPU latency column comparable.

- **weight_data_kib**: the raw weights (parameters × 4 bytes).
- **inference_file_kib**: the `.keras` file with no optimizer state, which is what a device would store.

In [ ]:
# Q6 cost measurements: every model built exactly as its training script builds it, measured
# in this one session so CPU latency is comparable. Cost does not depend on the trained weight
# values, so untrained copies (weights=None) give the same parameters, size, MACs and latency.
import tempfile
import keras
import pandas as pd
from src.evaluation import estimate_flops, estimate_macs, measure_cpu_latency_ms
from src.models.model_b import build_model_b


def sota_model(name, size=64):
    inputs = keras.Input(shape=(size, size, 3))
    if name == "MobileNetV2":       # as in MobileNetV2/model.py
        base = keras.applications.MobileNetV2(include_top=False, weights=None, input_shape=(size, size, 3), alpha=1.0)
        x = keras.layers.Rescaling(1.0 / 127.5, offset=-1)(inputs)
    else:                           # as in EfficientNetV2-B0/model.py
        base = keras.applications.EfficientNetV2B0(include_top=False, weights=None, input_shape=(size, size, 3),
                                                   include_preprocessing=True)
        x = inputs
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.2)(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name=name)


def cost(model):
    with tempfile.TemporaryDirectory() as d:
        path = f"{d}/inference.keras"
        model.save(path)                                   # uncompiled: no optimizer state
        inference_kib = Path(path).stat().st_size / 1024
    flops = estimate_flops(model)
    return {
        "input": f"{model.input_shape[1]}x{model.input_shape[2]}x{model.input_shape[3]}",
        "total_parameters": int(model.count_params()),
        "weight_data_kib": round(sum(w.numpy().nbytes for w in model.weights) / 1024, 1),
        "inference_file_kib": round(inference_kib, 1),
        "macs_profiler": estimate_macs(model, flops=flops),
        "mflops": round(flops / 1e6, 2) if flops else None,
        "cpu_latency_ms": round(measure_cpu_latency_ms(model), 3),
    }


models = {
    "Model B (28x28, native)": build_model_b(input_shape=(28, 28, 1)),
    "Model B (64x64, upscaled)": build_model_b(input_shape=(64, 64, 1)),
    "MobileNetV2 (64x64)": sota_model("MobileNetV2"),
    "EfficientNetV2-B0 (64x64)": sota_model("EfficientNetV2-B0"),
}
cost_rows = []
for label, model in models.items():
    print("measuring", label)
    cost_rows.append({"model": label, **cost(model)})
costs = pd.DataFrame(cost_rows)
costs.to_csv(DRIVE_ROOT / "q6_cost_measurements.csv", index=False)
costs

## 11. Q6 comparison table

Accuracy comes from each model's results:
- **Model B (28×28)**: the Q4 run in `results/model_b/model_b_rmsprop/` (RMSprop, lr 0.003, 60 epochs, final epoch).
- **Model B (64×64)**: the run selected above.
- **MobileNetV2 / EfficientNetV2-B0**: the first `results/**/config.json` with that model name **and `input_size` 64**, plus its `metrics.json`. Rows show "not available" until those 64×64 results are committed. Rerun this cell after pulling them.

In [ ]:
!git pull -q origin {BRANCH}
repo = Path("/content/CNN-for-Image-Classification")

def first(d, *keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

def sota_accuracy(name_part):
    for cfg_path in sorted(repo.glob("results/**/config.json")):
        cfg = json.loads(cfg_path.read_text())
        if name_part.lower() in str(cfg.get("model_name", "")).lower() and int(cfg.get("input_size", 0)) == 64:
            # Test results sit in final_test/ (possibly a run subfolder) or next to config.json.
            candidates = sorted((cfg_path.parent / "final_test").rglob("metrics.json")) + [cfg_path.parent / "metrics.json"]
            for metrics_path in candidates:
                if not metrics_path.exists():
                    continue
                met = json.loads(metrics_path.read_text())
                acc = met.get("test_accuracy")       # test-set files only, never validation
                if acc is not None:
                    return {"test_accuracy": acc, "macro_f1": met.get("macro_f1"),
                            "source": str(metrics_path.relative_to(repo))}
    return {"test_accuracy": None, "macro_f1": None, "source": "not available at 64x64 yet"}

b28 = json.loads((repo / "results/model_b/model_b_rmsprop/evaluation.json").read_text())
b64 = json.loads((DRIVE_ROOT / BEST / "metrics" / "evaluation.json").read_text())
accuracy = {
    "Model B (28x28, native)": {"test_accuracy": b28["test_accuracy"], "macro_f1": b28["macro_f1"],
                                "source": "results/model_b/model_b_rmsprop (Q4, final epoch)"},
    "Model B (64x64, upscaled)": {"test_accuracy": b64["test_accuracy"], "macro_f1": b64["macro_f1"],
                                  "source": f"{BEST} (best validation-loss epoch {b64['selected_epoch']})"},
    "MobileNetV2 (64x64)": sota_accuracy("MobileNet"),
    "EfficientNetV2-B0 (64x64)": sota_accuracy("EfficientNet"),
}
q6 = costs.copy()
q6.insert(2, "test_accuracy", [accuracy[m]["test_accuracy"] for m in q6.model])
q6.insert(3, "macro_f1", [accuracy[m]["macro_f1"] for m in q6.model])
q6["accuracy_source"] = [accuracy[m]["source"] for m in q6.model]
ref = q6.set_index("model").loc["Model B (64x64, upscaled)"]
q6["params_vs_B64"] = (q6.total_parameters / ref.total_parameters).round(1)
q6["macs_vs_B64"] = (q6.macs_profiler / ref.macs_profiler).round(1)
q6["latency_vs_B64"] = (q6.cpu_latency_ms / ref.cpu_latency_ms).round(1)
q6.to_csv(DRIVE_ROOT / "q6_comparison.csv", index=False)
q6

## 12. Package and download
Copies only the small files (no `.keras` weights; those stay in Drive) and downloads one zip. Put it in your Downloads folder so it can be committed to `results/model_b_64/`.

In [ ]:
import shutil
pkg = Path("/content/model_b64_results/results/model_b_64")
if pkg.exists(): shutil.rmtree(pkg)
pkg.mkdir(parents=True)

for path in DRIVE_ROOT.glob("model_b64_*/metrics/evaluation.json"):
    run = path.parent.parent
    dst = pkg / "all_runs" / run.name; dst.mkdir(parents=True)
    for f in ["evaluation.json", "history.csv", "architecture.json", "reporting_metrics.json"]:
        shutil.copy(run / "metrics" / f, dst / f)
shutil.copytree(DRIVE_ROOT / BEST / "report_figures", pkg / "selected" / "report_figures")
for f in ["evaluation.json", "history.csv", "architecture.json", "reporting_metrics.json"]:
    shutil.copy(DRIVE_ROOT / BEST / "metrics" / f, pkg / "selected" / f)
(pkg / "selected" / "run_name.txt").write_text(BEST + "\n")
for f in ["model_b64_runs.csv", "q6_cost_measurements.csv", "q6_comparison.csv"]:
    shutil.copy(DRIVE_ROOT / f, pkg / f)

archive = shutil.make_archive("/content/model_b64_results", "zip", "/content/model_b64_results")
!find /content/model_b64_results -type f | sort
from google.colab import files
files.download(archive)